## 0. Colab Setup (Mount Google Drive & Extract Dataset)

### How to run on Colab:
1. Compress the `brain_tumor_dataset` directory on your local machine to a `zip` file (ensure it is named `brain_tumor_dataset.zip`).
2. Upload `brain_tumor_dataset.zip` to your **Google Drive** (in the root directory).
3. Run the following cell to mount your Google Drive and extract the dataset.

In [ ]:
## Mount Google Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Google Drive mounted successfully!')
except:
    IN_COLAB = False
    print('Not running in Google Colab.')

## Unzip the dataset from Google Drive
if IN_COLAB:
    zip_path = '/content/drive/MyDrive/brain_tumor_dataset.zip'
    if os.path.exists(zip_path):
        print('Extracting dataset...')
        !unzip -q "{zip_path}" -d /content/
        print('Dataset extracted to /content/brain_tumor_dataset')
    else:
        print(f'Error: Could not find {zip_path}. Please upload the zip file to your Google Drive.')


## 1. Import the main libraries

In [ ]:
## Major Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

## Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

## Other
import os, cv2, warnings
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

## 2. Load data and look at the big picture

In [ ]:
## Set paths
if 'IN_COLAB' in globals() and IN_COLAB:
    DATASET_DIR = '/content/brain_tumor_dataset'
else:
    DATASET_DIR = os.path.join(os.getcwd(), 'brain_tumor_dataset')

TRAIN_DIR = os.path.join(DATASET_DIR, 'Training')
TEST_DIR = os.path.join(DATASET_DIR, 'Testing')

## Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.0001
NUM_CLASSES = 4
CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
print('Classes:', CLASS_NAMES)

In [ ]:
## Count images per class
data_info = []
for split in ['Training', 'Testing']:
    split_dir = os.path.join(DATASET_DIR, split)
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if os.path.isdir(cls_dir):
            count = len(os.listdir(cls_dir))
            data_info.append({'Split': split, 'Class': cls, 'Count': count})

df_info = pd.DataFrame(data_info)
df_info

* > The dataset has **5600 training** and **1600 testing** images
* > There are **4 classes**: glioma, meningioma, notumor, pituitary
* > The dataset is **perfectly balanced** (equal samples per class)

## 3. Exploratory Data Analysis (EDA)

### Class Distribution

In [ ]:
## Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, split in enumerate(['Training', 'Testing']):
    subset = df_info[df_info['Split'] == split]
    colors = ['#FF6B6B', '#FFA502', '#2ED573', '#1E90FF']
    bars = axes[idx].bar(subset['Class'], subset['Count'], color=colors, edgecolor='black', alpha=0.85)
    axes[idx].set_title(f'{split} Set Distribution', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Class', fontsize=12)
    axes[idx].set_ylabel('Count', fontsize=12)
    for bar, val in zip(bars, subset['Count']):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                       str(val), ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

### Sample Images

In [ ]:
## Display sample images from each class
fig, axes = plt.subplots(4, 5, figsize=(16, 12))
fig.suptitle('Sample Brain MRI Images by Class', fontsize=16, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    images = os.listdir(cls_dir)[:5]
    for col, img_name in enumerate(images):
        img_path = os.path.join(cls_dir, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        axes[row][col].imshow(img)
        axes[row][col].set_title(cls, fontsize=10)
        axes[row][col].axis('off')

plt.tight_layout()
plt.show()

### Image Size Analysis

In [ ]:
## Analyze image dimensions
widths, heights = [], []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    for img_name in os.listdir(cls_dir)[:100]:  ## sample 100 per class
        img = cv2.imread(os.path.join(cls_dir, img_name))
        if img is not None:
            h, w = img.shape[:2]
            widths.append(w)
            heights.append(h)

print(f'Width  - Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}')
print(f'Height - Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Image Width Distribution', fontsize=13)
axes[0].set_xlabel('Width (pixels)')
axes[1].hist(heights, bins=30, color='coral', edgecolor='black')
axes[1].set_title('Image Height Distribution', fontsize=13)
axes[1].set_xlabel('Height (pixels)')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Augmentation

In [ ]:
## Training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

## Testing data - only rescaling
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

## Create generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=42
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=42
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'Training samples: {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')
print(f'Test samples: {test_generator.samples}')
print(f'Class indices: {train_generator.class_indices}')

### Visualize Augmented Images

In [ ]:
## Show augmented samples
images, labels = next(train_generator)
class_labels = list(train_generator.class_indices.keys())

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Augmented Training Images', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        ax.set_title(class_labels[np.argmax(labels[i])], fontsize=11)
        ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Build the Model (Transfer Learning - VGG16)

In [ ]:
## Load VGG16 pre-trained on ImageNet
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

## Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

## Unfreeze last 4 layers for fine-tuning
for layer in base_model.layers[-4:]:
    layer.trainable = True

## Build classification head
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

## Compile
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Train the Model

In [ ]:
## Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('models/brain_tumor_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

## Train
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

## 7. Training History Visualization

In [ ]:
## Plot accuracy and loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

## Accuracy
axes[0].plot(history.history['accuracy'], label='Training', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

## Loss
axes[1].plot(history.history['loss'], label='Training', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluate the Model

In [ ]:
## Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

## 9. Confusion Matrix & Classification Report

In [ ]:
## Get predictions
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_names = list(test_generator.class_indices.keys())

## Classification Report
print('Classification Report:')
print('=' * 60)
print(classification_report(true_classes, predicted_classes, target_names=class_names, digits=4))

## Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax, linewidths=0.5)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.show()

## 10. Save the Model

In [ ]:
## Save the trained model
os.makedirs('models', exist_ok=True)
model.save('models/brain_tumor_model.h5')
print('Model saved locally to: models/brain_tumor_model.h5')

## If in Colab, also copy the model to Google Drive
if 'IN_COLAB' in globals() and IN_COLAB:
    import shutil
    drive_model_dir = '/content/drive/MyDrive/BrainTumorModels'
    os.makedirs(drive_model_dir, exist_ok=True)
    drive_model_path = os.path.join(drive_model_dir, 'brain_tumor_model.h5')
    shutil.copy('models/brain_tumor_model.h5', drive_model_path)
    print(f'\n✅ Model safely copied to Google Drive at: {drive_model_path}')
    print('👉 You can now download the model from Google Drive and place it in the models directory to run the Web App')

print(f'\nFinal Test Accuracy: {test_accuracy*100:.2f}%')
print('To run web interface locally: python app.py')

## 11. Test with a Single Image

In [ ]:
## Test prediction on a single image
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def predict_single_image(img_path):
    img = load_img(img_path, target_size=IMG_SIZE)
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    pred = model.predict(img_array, verbose=0)
    pred_class = class_names[np.argmax(pred[0])]
    confidence = np.max(pred[0]) * 100
    return pred_class, confidence, pred[0]

## Test on random images from test set
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Predictions on Test Images', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    cls = class_names[i % len(class_names)]
    cls_dir = os.path.join(TEST_DIR, cls)
    img_name = os.listdir(cls_dir)[i]
    img_path = os.path.join(cls_dir, img_name)
    
    pred_class, confidence, _ = predict_single_image(img_path)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    color = 'green' if pred_class == cls else 'red'
    ax.set_title(f'True: {cls}\nPred: {pred_class} ({confidence:.1f}%)', fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()